# Чистовик: CatBoost, квантили 0.1 / 0.5 / 0.9

Читает `data/`, считает CV по фолдам (тип разбиения выбирается под структуру теста), пишет `submission.csv` и `cv.json` и печатает `cv.json` последней строкой.

In [ ]:
import json, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, GroupKFold
from catboost import CatBoostRegressor

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")
TARGET = "daily_orders"
y = tr[TARGET].to_numpy(float)
QUANTILES = (0.1, 0.5, 0.9)
DATE = next((c for c in tr.columns if "date" in c.lower()), None)


def make_features(df, d0):
    X = df.drop(columns=[c for c in ("id", TARGET) if c in df.columns]).copy()
    if DATE is not None:
        d = pd.to_datetime(X[DATE])
        X["dow"] = d.dt.dayofweek
        X["month"] = d.dt.month
        X["day"] = d.dt.day
        X["doy"] = d.dt.dayofyear
        X["t"] = (d - d0).dt.days
        X = X.drop(columns=[DATE])
    for c in X.columns:
        if X[c].dtype == object:
            X[c] = X[c].fillna("NA").astype(str)
    return X


d0 = pd.to_datetime(tr[DATE]).min() if DATE is not None else None
Xtr = make_features(tr, d0)
Xte = make_features(te, d0)
CATS = [c for c in Xtr.columns if Xtr[c].dtype == object]


def predict(Xa, ya, Xb):
    """Три квантильные модели CatBoost на log1p(y), затем монотонность и клип."""
    out = []
    for a in QUANTILES:
        m = CatBoostRegressor(
            loss_function=f"Quantile:alpha={a}",
            iterations=800,
            learning_rate=0.05,
            depth=6,
            random_seed=0,
            verbose=0,
            cat_features=CATS,
        )
        m.fit(Xa, np.log1p(ya))
        out.append(np.expm1(m.predict(Xb)))
    q = np.sort(np.column_stack(out), axis=1)
    return np.clip(q, 0, 5000)


# CV: разбиение подбираем под то, как устроен тест
splits = None
if DATE is not None and pd.to_datetime(te[DATE]).min() > pd.to_datetime(tr[DATE]).max():
    # тест целиком в будущем -> expanding window по времени
    r = pd.to_datetime(tr[DATE]).rank(method="first").to_numpy()
    edges = np.quantile(r, [0.4, 0.55, 0.7, 0.85, 1.0])
    splits = []
    bounds = [0] + list(edges)
    for k in range(1, len(bounds) - 1):
        a = np.where(r <= bounds[k])[0]
        b = np.where((r > bounds[k]) & (r <= bounds[k + 1]))[0]
        splits.append((a, b))
    print("CV: time expanding window")
elif "city" in tr.columns and len(set(te["city"]) & set(tr["city"])) == 0:
    # в тесте новые города -> группируем по городу
    splits = list(GroupKFold(4).split(tr, groups=tr["city"]))
    print("CV: GroupKFold by city")
else:
    splits = list(KFold(4, shuffle=True, random_state=0).split(tr))
    print("CV: random KFold")

folds = []
for a, b in splits:
    p = predict(Xtr.iloc[a], y[a], Xtr.iloc[b])
    folds.append(100 * quantile_quality(y[b], p[:, 0], p[:, 1], p[:, 2], tr["city"].iloc[b]))
print("folds:", [round(f, 3) for f in folds])

# посылка: колонки и строки как в sample_submission.csv, 0 <= p10 <= p50 <= p90 <= 5000
test = np.round(predict(Xtr, y, Xte), 1)
sub = pd.DataFrame({"id": te["id"],
                    "predicted_daily_orders_p10": test[:, 0],
                    "predicted_daily_orders_p50": test[:, 1],
                    "predicted_daily_orders_p90": test[:, 2]})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"]) and (sub.iloc[:, 1:].to_numpy() >= 0).all()
sub.to_csv("submission.csv", index=False)

cv = cv_json("catboost", [f / 100 for f in folds])   # пишет cv.json
print(json.dumps(cv))                                  # последняя строка — для проверки воспроизводимости
